---
title: Motion Estimation
sidebarTitle: Motion Estimation
description: Motion estimation from intuition to implementation using synthetic image pairs, patch matching, sparse motion fields, and failure analysis.
---


# Motion estimation

Motion estimation asks a simple question: if something moves between two frames, can we recover **where it went**?

In this notebook we will:

- build intuition for motion in image coordinates,
- recover motion with a readable patch-matching baseline,
- estimate a sparse motion field on synthetic data with known ground truth,
- quantify accuracy with endpoint error and match ratios,
- study how patch size, search radius, and noise change the results,
- and inspect failure modes that motivate more advanced optical-flow methods.

The goal is not to be fancy. The goal is to make the core idea visible and trustworthy.


## Intuition

Between two video frames, a moving object often keeps a similar local appearance.
That means a small patch around a point in Frame 1 may reappear a few pixels away in Frame 2.

We will use the image-coordinate convention common in vision:

- $x$ increases to the **right**,
- $y$ increases **downward**,
- a motion vector is written as $(dx, dy)$.

So $(dx=5, dy=-4)$ means "move 5 pixels right and 4 pixels up."

![Frame pair overview](assets/frame-pair.svg)


## Imports and plotting style

The notebook uses only `numpy` and `matplotlib`. A fixed random seed keeps the synthetic examples reproducible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#fbfdff",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)

rng = np.random.default_rng(7)


## Synthetic example setup

We start with clean synthetic frames because the true motion is known exactly.
That lets us debug the estimator before dealing with real video.

The scene uses a few textured geometric shapes so patch matching has enough visual structure to lock onto.


The helper below builds a frame, applies an integer translation, and optionally adds Gaussian noise. Running the cell also draws the two frames side by side.


In [ ]:
def normalize_image(img):
    return np.clip(img, 0.0, 1.0)


def add_rectangle(img, x0, y0, w, h, base=0.75, texture="flat"):
    yy, xx = np.mgrid[y0 : y0 + h, x0 : x0 + w]
    patch = np.full((h, w), base, dtype=float)
    if texture == "gradient":
        patch *= 0.55 + 0.45 * (xx - x0) / max(w - 1, 1)
    elif texture == "checker":
        patch *= 0.65 + 0.35 * (((xx - x0) // 4 + (yy - y0) // 4) % 2)
    img[y0 : y0 + h, x0 : x0 + w] = np.maximum(img[y0 : y0 + h, x0 : x0 + w], patch)


def add_disk(img, cx, cy, radius, base=0.82):
    yy, xx = np.mgrid[: img.shape[0], : img.shape[1]]
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    mask = dist <= radius
    ripple = 0.78 + 0.22 * np.cos(dist[mask] / max(radius, 1) * np.pi)
    img[mask] = np.maximum(img[mask], base * ripple)


def shift_image_integer(img, dx, dy, fill_value=0.06):
    shifted = np.full_like(img, fill_value)
    h, w = img.shape

    src_x0 = max(0, -dx)
    src_x1 = min(w, w - dx)
    src_y0 = max(0, -dy)
    src_y1 = min(h, h - dy)

    dst_x0 = max(0, dx)
    dst_x1 = dst_x0 + (src_x1 - src_x0)
    dst_y0 = max(0, dy)
    dst_y1 = dst_y0 + (src_y1 - src_y0)

    if src_x1 > src_x0 and src_y1 > src_y0:
        shifted[dst_y0:dst_y1, dst_x0:dst_x1] = img[src_y0:src_y1, src_x0:src_x1]
    return shifted


def make_synthetic_pair(dx=5, dy=-4, noise_std=0.0, size=96):
    frame1 = np.full((size, size), 0.06, dtype=float)
    add_rectangle(frame1, 14, 18, 24, 20, base=0.88, texture="gradient")
    add_rectangle(frame1, 52, 14, 22, 26, base=0.82, texture="checker")
    add_rectangle(frame1, 22, 62, 28, 12, base=0.70, texture="checker")
    add_disk(frame1, 68, 66, 11, base=0.95)

    frame2 = shift_image_integer(frame1, dx=dx, dy=dy, fill_value=0.06)

    if noise_std > 0:
        frame1 = normalize_image(frame1 + rng.normal(0.0, noise_std, frame1.shape))
        frame2 = normalize_image(frame2 + rng.normal(0.0, noise_std, frame2.shape))

    return frame1, frame2, np.array([dx, dy], dtype=int)


def show_frame_pair(frame1, frame2, gt_motion):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)
    titles = ["Frame 1", f"Frame 2 (ground truth motion = {tuple(gt_motion)})"]
    for ax, image, title in zip(axes, [frame1, frame2], titles):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1, origin="upper")
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.show()


frame1, frame2, gt_motion = make_synthetic_pair(dx=5, dy=-4, noise_std=0.015)
show_frame_pair(frame1, frame2, gt_motion)


## Patch matching / local search

The baseline method is deliberately simple:

1. extract an odd-sized patch around a point in Frame 1,
2. search a square window around the same location in Frame 2,
3. score each candidate with SSD (sum of squared differences),
4. keep the displacement with the lowest score.

This is easy to read and easy to reason about, which makes it a good teaching baseline.

![Patch matching diagram](assets/patch-matching.svg)


The next cell defines reusable helpers for patch extraction, SSD scoring, exhaustive local search, and patch-match visualization.


In [ ]:
def extract_patch(image, center_xy, patch_size):
    radius = patch_size // 2
    x, y = map(int, center_xy)
    if x - radius < 0 or y - radius < 0:
        return None
    if x + radius >= image.shape[1] or y + radius >= image.shape[0]:
        return None
    return image[y - radius : y + radius + 1, x - radius : x + radius + 1]


def ssd_cost(template, candidate):
    diff = template - candidate
    return float(np.sum(diff * diff))


def estimate_patch_motion(frame1, frame2, point_xy, patch_size=11, search_radius=8):
    template = extract_patch(frame1, point_xy, patch_size)
    if template is None:
        raise ValueError("Source patch falls outside the image.")

    costs = np.full((2 * search_radius + 1, 2 * search_radius + 1), np.inf)
    best = {"cost": np.inf, "dx": 0, "dy": 0, "match_patch": None, "best_point": point_xy}

    for row, dy in enumerate(range(-search_radius, search_radius + 1)):
        for col, dx in enumerate(range(-search_radius, search_radius + 1)):
            candidate_point = (point_xy[0] + dx, point_xy[1] + dy)
            candidate = extract_patch(frame2, candidate_point, patch_size)
            if candidate is None:
                continue
            cost = ssd_cost(template, candidate)
            costs[row, col] = cost
            if cost < best["cost"]:
                best = {
                    "cost": cost,
                    "dx": dx,
                    "dy": dy,
                    "match_patch": candidate,
                    "best_point": candidate_point,
                }

    best["template_patch"] = template
    best["cost_map"] = costs
    return best


def draw_patch_matching_demo(frame1, frame2, point_xy, match_result, patch_size, search_radius):
    radius = patch_size // 2
    x, y = point_xy
    best_x, best_y = match_result["best_point"]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)

    axes[0].imshow(frame1, cmap="gray", vmin=0, vmax=1, origin="upper")
    axes[0].add_patch(Rectangle((x - radius, y - radius), patch_size, patch_size,
                                edgecolor="#355c7d", facecolor="none", linewidth=2.5))
    axes[0].set_title("Frame 1: source patch")
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    axes[1].imshow(frame2, cmap="gray", vmin=0, vmax=1, origin="upper")
    axes[1].add_patch(Rectangle((x - search_radius - radius, y - search_radius - radius),
                                2 * search_radius + patch_size,
                                2 * search_radius + patch_size,
                                edgecolor="#7c3aed", facecolor="none", linewidth=2.0, linestyle="--"))
    axes[1].add_patch(Rectangle((best_x - radius, best_y - radius), patch_size, patch_size,
                                edgecolor="#d94841", facecolor="none", linewidth=2.5))
    axes[1].annotate("", xy=(best_x, best_y), xytext=(x, y),
                     arrowprops=dict(arrowstyle="->", lw=2.2, color="#1f77b4"))
    axes[1].set_title("Frame 2: search window and best match")
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    im = axes[2].imshow(match_result["cost_map"], cmap="magma_r", origin="lower")
    axes[2].set_title("SSD cost over candidate displacements")
    axes[2].set_xlabel("dx")
    axes[2].set_ylabel("dy")
    tick_positions = np.arange(0, 2 * search_radius + 1, 2)
    tick_labels = np.arange(-search_radius, search_radius + 1, 2)
    axes[2].set_xticks(tick_positions, tick_labels)
    axes[2].set_yticks(tick_positions, tick_labels)
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    plt.show()


## Single-point motion estimation demo

We first track one carefully chosen point so every part of the process is visible.
The figure highlights the source patch, the search window, the matched patch, and the resulting motion vector.


In [ ]:
point_xy = (63, 25)
patch_size = 11
search_radius = 8

match_result = estimate_patch_motion(frame1, frame2, point_xy, patch_size=patch_size, search_radius=search_radius)
print(f"Estimated motion at {point_xy}: (dx={match_result['dx']}, dy={match_result['dy']})")
print(f"Ground truth motion: {tuple(gt_motion)}")

draw_patch_matching_demo(frame1, frame2, point_xy, match_result, patch_size, search_radius)

fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.1), constrained_layout=True)
axes[0].imshow(match_result["template_patch"], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Source patch")
axes[1].imshow(match_result["match_patch"], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Matched patch")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()


## Sparse motion field estimation

One point is useful for intuition, but a motion field becomes more interesting when we estimate many arrows.
We will sample textured points on a grid and run the same local matcher at each one.

![Sparse motion field](assets/sparse-field.svg)


In [ ]:
def sample_textured_points(frame, patch_size, stride=12, threshold=0.18):
    radius = patch_size // 2
    points = []
    for y in range(radius + 2, frame.shape[0] - radius - 2, stride):
        for x in range(radius + 2, frame.shape[1] - radius - 2, stride):
            patch = extract_patch(frame, (x, y), patch_size)
            if patch is None:
                continue
            if patch.std() > threshold or patch.mean() > 0.35:
                points.append((x, y))
    return points


def estimate_sparse_motion(frame1, frame2, points, patch_size=11, search_radius=8):
    estimates = []
    for point in points:
        result = estimate_patch_motion(frame1, frame2, point, patch_size=patch_size, search_radius=search_radius)
        estimates.append({
            "point": point,
            "dx": result["dx"],
            "dy": result["dy"],
            "cost": result["cost"],
        })
    return estimates


def summarize_motion_metrics(estimates, gt_motion):
    vectors = np.array([[item["dx"], item["dy"]] for item in estimates], dtype=float)
    gt = np.asarray(gt_motion, dtype=float)
    errors = vectors - gt[None, :]
    epe = np.sqrt(np.sum(errors ** 2, axis=1))
    exact = np.all(vectors == gt[None, :], axis=1)
    near = epe <= 1.0
    return {
        "num_points": len(estimates),
        "mean_epe": float(epe.mean()),
        "median_epe": float(np.median(epe)),
        "max_epe": float(epe.max()),
        "exact_match_ratio": float(exact.mean()),
        "within_1px_ratio": float(near.mean()),
    }


points = sample_textured_points(frame1, patch_size=11, stride=12, threshold=0.06)
estimates = estimate_sparse_motion(frame1, frame2, points, patch_size=11, search_radius=8)
summary = summarize_motion_metrics(estimates, gt_motion)
summary

fig, ax = plt.subplots(figsize=(6.6, 6.0), constrained_layout=True)
ax.imshow(frame1, cmap="gray", vmin=0, vmax=1, origin="upper")
x_coords = np.array([item["point"][0] for item in estimates])
y_coords = np.array([item["point"][1] for item in estimates])
dx = np.array([item["dx"] for item in estimates])
dy = np.array([item["dy"] for item in estimates])
ax.quiver(x_coords, y_coords, dx, dy, color="#0f4c81", angles="xy", scale_units="xy", scale=1)
ax.set_title("Sparse motion field estimated by patch matching")
ax.set_xticks([])
ax.set_yticks([])
plt.show()


## Validation metrics

Because the synthetic translation is known, we can score the estimates directly.

- **Endpoint error (EPE)** is the Euclidean distance between the estimated vector and the true vector.
- **Exact match ratio** is the fraction of estimates that recover the exact integer motion.
- **Correct-within-1-pixel ratio** is a softer measure that counts near misses as acceptable.

These metrics answer slightly different questions, so it is helpful to look at all of them.


In [ ]:
metric_names = ["mean_epe", "median_epe", "max_epe", "exact_match_ratio", "within_1px_ratio"]
for key in metric_names:
    print(f"{key:>20}: {summary[key]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.4), constrained_layout=True)
axes[0].bar(["dx", "dy"], gt_motion, color=["#4c78a8", "#f58518"], alpha=0.9, label="ground truth")
mean_estimate = np.array([
    np.mean([item["dx"] for item in estimates]),
    np.mean([item["dy"] for item in estimates]),
])
axes[0].plot(["dx", "dy"], mean_estimate, marker="o", linewidth=2, color="#1f2933", label="mean estimate")
axes[0].set_title("Average estimated motion vs ground truth")
axes[0].legend(frameon=False)

ratio_values = [summary["exact_match_ratio"], summary["within_1px_ratio"]]
axes[1].bar(["exact", "within 1 px"], ratio_values, color=["#54a24b", "#72b7b2"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Match-quality ratios")
plt.show()


## Parameter sweeps / analysis

Three knobs matter immediately:

- **Patch size**: larger patches are often more stable, but they blur away local detail.
- **Search radius**: larger windows can recover larger motion, but they cost more and may invite distractors.
- **Noise level**: noisy images reduce the reliability of appearance matching.

![Parameter tradeoffs](assets/parameter-tradeoffs.svg)


In [ ]:
def evaluate_setting(dx=5, dy=-4, noise_std=0.015, patch_size=11, search_radius=8):
    f1, f2, gt = make_synthetic_pair(dx=dx, dy=dy, noise_std=noise_std)
    pts = sample_textured_points(f1, patch_size=patch_size, stride=12, threshold=0.06)
    est = estimate_sparse_motion(f1, f2, pts, patch_size=patch_size, search_radius=search_radius)
    return summarize_motion_metrics(est, gt)


patch_sizes = [7, 11, 15, 21]
search_radii = [2, 4, 6, 8, 10]
noise_levels = [0.00, 0.01, 0.03, 0.06, 0.10]

patch_results = [evaluate_setting(patch_size=p, search_radius=8, noise_std=0.015) for p in patch_sizes]
radius_results = [evaluate_setting(patch_size=11, search_radius=r, noise_std=0.015) for r in search_radii]
noise_results = [evaluate_setting(patch_size=11, search_radius=8, noise_std=n) for n in noise_levels]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)

axes[0].plot(patch_sizes, [item["mean_epe"] for item in patch_results], marker="o", color="#2d6a4f")
axes[0].set_title("Patch size vs mean EPE")
axes[0].set_xlabel("patch size")
axes[0].set_ylabel("mean EPE")

axes[1].plot(search_radii, [item["within_1px_ratio"] for item in radius_results], marker="o", color="#355c7d")
axes[1].set_title("Search radius vs within-1px ratio")
axes[1].set_xlabel("search radius")
axes[1].set_ylabel("correct ratio")
axes[1].set_ylim(0, 1.05)

axes[2].plot(noise_levels, [item["mean_epe"] for item in noise_results], marker="o", color="#bc4749")
axes[2].set_title("Noise level vs mean EPE")
axes[2].set_xlabel("noise std")
axes[2].set_ylabel("mean EPE")

plt.show()


## Failure cases

A basic local matcher has clear blind spots. We will demonstrate two of the most important ones.

1. **Repetitive texture**: many candidate patches look equally good.
2. **Motion outside the search radius**: the correct answer is never even evaluated.

![Failure cases](assets/failure-cases.svg)


In [ ]:
def make_repetitive_texture_pair(dx=4, dy=0, size=96):
    frame1 = np.full((size, size), 0.08, dtype=float)
    for x0 in range(18, 78, 8):
        frame1[:, x0 : x0 + 4] = 0.72
    frame2 = shift_image_integer(frame1, dx=dx, dy=dy, fill_value=0.08)
    return frame1, frame2, np.array([dx, dy], dtype=int)


def failure_demo(frame1, frame2, point_xy, gt_motion, patch_size, search_radius, title):
    result = estimate_patch_motion(frame1, frame2, point_xy, patch_size=patch_size, search_radius=search_radius)
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), constrained_layout=True)
    axes[0].imshow(frame1, cmap="gray", vmin=0, vmax=1, origin="upper")
    axes[0].add_patch(Rectangle((point_xy[0] - patch_size // 2, point_xy[1] - patch_size // 2),
                                patch_size, patch_size, edgecolor="#355c7d", facecolor="none", linewidth=2.2))
    axes[0].set_title(f"{title}: Frame 1")
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    axes[1].imshow(frame2, cmap="gray", vmin=0, vmax=1, origin="upper")
    axes[1].add_patch(Rectangle((point_xy[0] - search_radius - patch_size // 2,
                                 point_xy[1] - search_radius - patch_size // 2),
                                2 * search_radius + patch_size,
                                2 * search_radius + patch_size,
                                edgecolor="#7c3aed", facecolor="none", linewidth=2.0, linestyle="--"))
    axes[1].set_title(f"search window (radius={search_radius})")
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    im = axes[2].imshow(result["cost_map"], cmap="magma_r", origin="lower")
    axes[2].set_title(f"estimated motion = ({result['dx']}, {result['dy']}); gt = {tuple(gt_motion)}")
    axes[2].set_xlabel("dx")
    axes[2].set_ylabel("dy")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    plt.show()


repetitive_f1, repetitive_f2, repetitive_gt = make_repetitive_texture_pair(dx=4, dy=0)
failure_demo(repetitive_f1, repetitive_f2, point_xy=(44, 48), gt_motion=repetitive_gt,
             patch_size=11, search_radius=8, title="Repetitive texture")

large_f1, large_f2, large_gt = make_synthetic_pair(dx=11, dy=-7, noise_std=0.01)
failure_demo(large_f1, large_f2, point_xy=(63, 25), gt_motion=large_gt,
             patch_size=11, search_radius=5, title="Motion beyond the search radius")


## Limitations and takeaway

Patch matching works well when:

- motion is moderate,
- the search window covers the true displacement,
- local texture is distinctive,
- and noise is not too strong.

It struggles when texture is ambiguous, motion is too large, illumination changes, or the underlying motion varies strongly inside one patch.
Those limitations are exactly why more advanced methods such as Lucas-Kanade, Horn-Schunck style optical flow, pyramidal search, and learned flow estimators exist.


## Reproducibility

- All random choices use a fixed seed.
- The main synthetic example uses integer translation so the ground truth is unambiguous.
- The notebook is organized so you can run it top to bottom without hidden state.

If you want an extra exercise, try replacing the global translation with rotation or locally varying motion and see where the baseline breaks first.
